# ML-04 — Search Intelligence Data Contract

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/damlablgc/flyrankinternship/blob/main/work/notebooks/w03_data_contract.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Unit of analysis + time window

*One row = one what, over which dates? State it, then verify it below.*

## 2. Fields: feature / label / context / excluded

*Sort every field you plan to touch into these four buckets. Excluded needs a why.*

In [9]:
from google.colab import userdata
import duckdb

HF_TOKEN = userdata.get('HF_TOKEN')
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

REL = 'hf://datasets/FlyRank/internship-warehouse'
MONTH_PATH = f"{REL}/fact_content_daily_performance/month=2026-03/*.parquet"

In [10]:
grain_check = con.sql(f"""
    SELECT client_hash_id, content_hash_id, report_date, COUNT(*) AS c
    FROM read_parquet('{MONTH_PATH}')
    GROUP BY 1, 2, 3
    HAVING c > 1
    LIMIT 5
""").df()
print("Tekrar eden satır sayısı:", len(grain_check))
grain_check

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Tekrar eden satır sayısı: 0


,client_hash_id,content_hash_id,report_date,c


In [11]:
counts = con.sql(f"""
    SELECT
        COUNT(*)                        AS n_rows,
        MIN(report_date)                AS min_date,
        MAX(report_date)                AS max_date,
        COUNT(DISTINCT client_hash_id)   AS n_clients,
        COUNT(DISTINCT content_hash_id)  AS n_content
    FROM read_parquet('{MONTH_PATH}')
""").df()
counts

,n_rows,min_date,max_date,n_clients,n_content
0,9841378,2026-03-01,2026-03-31,55,331437


In [12]:
availability = con.sql(f"""
    SELECT
        COUNT(*) AS total_rows,
        SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) AS ga4_available_rows,
        ROUND(100.0 * SUM(CASE WHEN ga4_data_available IS TRUE THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_available
    FROM read_parquet('{MONTH_PATH}')
""").df()
availability

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,total_rows,ga4_available_rows,pct_available
0,9841378,413966.0,4.2


In [16]:
DECISION_DAY = '2026-03-15'

features = con.sql(f"""
    WITH fx AS (
        SELECT
            f.client_hash_id, f.content_hash_id,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_impressions ELSE 0 END) AS imp_trailing,
            SUM(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_clicks ELSE 0 END)      AS clk_trailing,
            AVG(CASE WHEN f.report_date <= DATE '{DECISION_DAY}' THEN f.gsc_avg_position END)       AS pos_trailing,
            MAX(CASE WHEN f.ga4_data_available IS TRUE THEN 1 ELSE 0 END)                            AS has_ga4_data
        FROM read_parquet('{MONTH_PATH}') f
        GROUP BY 1, 2
    )
    SELECT
        fx.*,
        DATE_DIFF('day', dc.content_created_date, DATE '{DECISION_DAY}') AS content_age_days
    FROM fx
    LEFT JOIN read_parquet('{REL}/dim_content.parquet') dc
      ON fx.content_hash_id = dc.content_hash_id
    WHERE dc.content_created_date <= DATE '{DECISION_DAY}'
""").df()

features.head()

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

,client_hash_id,content_hash_id,imp_trailing,clk_trailing,pos_trailing,has_ga4_data,content_age_days
0,client_73cda7b4e4f265ea,content_7a105f548d9c6916,4173.0,6.0,6.327311,1,380
1,client_73cda7b4e4f265ea,content_a3ea9792f793ec72,245.0,0.0,3.906852,0,380
2,client_73cda7b4e4f265ea,content_36c36abc7650d7af,3705.0,3.0,6.473735,1,380
3,client_73cda7b4e4f265ea,content_a7da352b73b02668,2440.0,8.0,7.259861,1,380
4,client_73cda7b4e4f265ea,content_f39be42b42a4e8f6,14.0,0.0,9.000000,1,380


Five features:
1. `imp_trailing` — sum of impressions up to March 15; only past days, nothing after.
2. `clk_trailing` — same reasoning, only days on or before the decision day.
3. `pos_trailing` — average search position from days already observed.
4. `has_ga4_data` — whether GA4 tracking existed as of that date, a fact of that moment.
5. `content_age_days` — computed from `content_created_date`, which is fixed and
   always at or before the decision day (rows where the page was created AFTER
   March 15 were excluded entirely).

## 3. Verify it with queries (grain, counts, missing values, windows)

*Every claim above gets a query cell here. A contract claim without a query next to it is a guess.*

## 4. Data limits

*What can this data never tell you? Unbalanced history, GSC-only early rows, window overlaps.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.